# 03. Equilibrium and Kinetic Profiles

An equilibrium is the answer to a question the plasma is always solving: where must the
current sit so that the magnetic force balances the pressure gradient everywhere at once.
This session takes a reconstructed VEST equilibrium apart, asks whether it actually solved
that equation, and then re-solves it with a dedicated code to see what changes.


## Session Overview

By the end of this session you will be able to:

- read an equilibrium's flux map, boundary and 0-D parameters, and say which of them the
  reconstruction measured and which VAFT derived;
- plot a profile against each of the radial coordinates VAFT supports, and say what each
  one is *for*;
- map flux surfaces onto a camera image and watch the boundary move;
- measure how well an equilibrium satisfies its own Grad-Shafranov equation;
- refine an equilibrium with CHEASE and vary its shape, beta and internal inductance.

Two shots are used, because no single packaged VEST discharge carries everything:

| shot | what it brings |
| --- | --- |
| 39915 | nine equilibrium slices through a discharge, plus 66 fast-camera frames |
| 48224 | kinetic profiles (Thomson, charge exchange) and a CHEASE-refined counterpart |


## Physical Context

### The Grad-Shafranov equation

In an axisymmetric plasma, force balance collapses to one scalar equation for the poloidal
flux $\psi(R, Z)$:

$$\Delta^* \psi \;=\; -\mu_0 R^2 \, p'(\psi) \;-\; F F'(\psi),
\qquad
\Delta^* \equiv R\frac{\partial}{\partial R}\!\left(\frac{1}{R}\frac{\partial}{\partial R}\right) + \frac{\partial^2}{\partial Z^2}$$

Everything else follows from $\psi$: the flux surfaces are its contours, the field is its
gradient, and the two free functions $p'$ and $FF'$ carry the pressure and the current.

### Reconstruction against forward modelling

Those are two different jobs, and VEST uses a different code for each.

**EFIT reconstructs.** It is given magnetic measurements — flux loops, field probes, coil
currents — and finds the $\psi$, $p'$ and $FF'$ most consistent with them. It answers *what
was the plasma doing*, and its answer is only as good as the diagnostics constrain it.

**CHEASE forward-models.** It is given a boundary and the two profiles, and solves the
equation on a fine mesh to high accuracy. It answers *what equilibrium do these inputs
imply*. It cannot tell you about a discharge it was not given.

They also differ in what is free. EFIT solves a **free-boundary** problem — the boundary is
part of the answer, and its grid extends past the plasma to the coils. CHEASE solves a
**fixed-boundary** problem — you hand it the boundary and it works only inside. That
difference matters later, when we measure both.


## Load / Prepare Data

### The reconstruction

`sample_ods()` returns the packaged 39915 discharge with no arguments and no network access.


In [ ]:
import vaft
import matplotlib.pyplot as plt


In [ ]:
ods = vaft.omas.sample_ods()
ods["equilibrium.time"]


Nine slices, from 316 to 331 ms. The last one is worth knowing about before you index into
it: the plasma current has reached zero and the reconstruction is degenerate — no boundary,
and `psi_axis == psi_boundary`. It is real output, not a bug, and a reminder that a stored
slice is not automatically a usable equilibrium.


In [ ]:
last = ods["equilibrium.time_slice"][8]
print("ip        :", float(last["global_quantities.ip"]))
print("psi_axis  :", float(last["global_quantities.psi_axis"]))
print("boundary  :", "boundary" in last)


### The kinetic shot

39915 was reconstructed from magnetics alone. Shot 48224 additionally has Thomson scattering
and ion Doppler spectroscopy, and ships with both its EFIT reconstruction and a
CHEASE-refined version of it. It lives in the repository rather than the installed package,
so this cell needs a source checkout.


In [ ]:
kinetic = vaft.omas.load(str(vaft.data.data_path("kineticEfit/ods_48224_300ms.json")))
sorted(kinetic.keys())


## Guided Analysis

### The flux map and the boundary

Start with the picture the whole session rests on.


In [ ]:
vaft.omas.plot_equilibrium_field_psi(ods, time_slice=3)
plt.show()


In [ ]:
vaft.omas.plot_equilibrium_geometry_boundary(ods, time_slice=3)
plt.show()


### The profiles the reconstruction produced

`q`, `pressure`, `p'` and `FF'` are the equilibrium's own description of itself.


In [ ]:
vaft.omas.plot_equilibrium_overview_profiles(ods, time_slice=3)
plt.show()


### What an EFIT file does *not* carry

Ask 39915 for its normalised beta or its internal inductance and VAFT refuses, because a
g-file stores only `psi`, `p`, `p'`, `F`, `FF'` and `q`. Everything else — volume, the
flux-surface averages, the shape parameters, beta, `li` — has to be *derived* by tracing the
flux surfaces.

That derivation is one call, and it is the most useful thing in this session to remember.


In [ ]:
names = {row["name"] for row in vaft.omas.available_plots(ods)}
print("beta_n available? ", "equilibrium_time_beta_n" in names)
print("li available?     ", "equilibrium_time_li" in names)
print("profiles_1d fields:", len(ods["equilibrium.time_slice.3.profiles_1d"]))


In [ ]:
vaft.omas.update_equilibrium_derived_profiles(ods)

names = {row["name"] for row in vaft.omas.available_plots(ods)}
print("beta_n available? ", "equilibrium_time_beta_n" in names)
print("li available?     ", "equilibrium_time_li" in names)
print("profiles_1d fields:", len(ods["equilibrium.time_slice.3.profiles_1d"]))


Seven fields became twenty-four, and two plots that did not exist a moment ago now do. This
is a general habit worth forming: **a plot VAFT refuses is often a quantity nobody has
derived yet, not a quantity the shot lacks.**

Two cautions about what you just computed:

- **`li` here is `li(3)`**, the ITER/Jackson definition $2\int B_p^2\,dV / (\mu_0^2 I_p^2 R_0)$.
  VAFT does not implement `li(1)` or `li(2)`, and they are not interchangeable.
- **"beta poloidal" names three different quantities** in common use, differing by up to 26%
  on VEST. VAFT stores the Data Dictionary one; a second, matching EFIT's own reported value,
  is available separately. Always say which you mean.


In [ ]:
vaft.omas.plot_equilibrium_overview_histories(ods)
plt.show()


### Radial coordinates

A 1-D profile needs an abscissa, and there is no single right one. VAFT offers five, and
`coordinate=` selects between them:

| coordinate | is | useful for |
| --- | --- | --- |
| `psi_norm` | normalised poloidal flux | anything derived from the flux map itself |
| `rho_tor_norm` | normalised toroidal flux radius | transport, where volume matters |
| `sqrt_phi_norm` | square root of normalised toroidal flux | profile shapes near the axis |
| `r_major` | outboard major radius | comparing against a midplane diagnostic |
| `r_minor` | minor radius | geometric reasoning |

The same `q` profile, drawn three ways:


In [ ]:
vaft.omas.plot_equilibrium_profile_q(ods, time_slice=3, coordinate="psi_norm")
plt.show()


In [ ]:
vaft.omas.plot_equilibrium_profile_q(ods, time_slice=3, coordinate="rho_tor_norm")
plt.show()


In [ ]:
vaft.omas.plot_equilibrium_profile_q(ods, time_slice=3, coordinate="r_major")
plt.show()


The curve is the same physics; only the ruler changed. Note that the axis label changes with
it — VAFT will not draw a curve against an abscissa it cannot honestly name, and falls back
to a coarser coordinate (or to the sample index) rather than mislabel one.


### Kinetic profiles

48224 has measured electron temperature and density. Thomson scattering gives a handful of
points; `core_profiles` carries the fitted profile through them.


In [ ]:
vaft.omas.plot_thomson_scattering_profile_electron_temperature(kinetic)
plt.show()


In [ ]:
vaft.omas.plot_electron_temperature_profile(kinetic)
plt.show()


In [ ]:
vaft.omas.plot_electron_density_profile(kinetic)
plt.show()


### Flux surfaces on a camera image

VEST's fast camera looks into the vessel, and the packaged 39915 sample ships 66 frames.
They are images, not an IDS, so they have to be mapped in before the plotting layer can use
them — two public mapper calls, one for the static description and one for the frames.


In [ ]:
import numpy as np
from PIL import Image
from vaft.data.resources import sample_camera_visible_frame_paths
from vaft.machine_mapping import camera_visible as cam

times_eq = np.asarray(ods["equilibrium.time"])
frames = [
    (t, path) for t, path in sample_camera_visible_frame_paths(39915)
    if times_eq.min() <= t <= times_eq.max()
]
images = [np.asarray(Image.open(path).convert("L")) for _, path in frames]

cam.vfit_camera_visible_static(ods, lines_n=images[0].shape[0], columns_n=images[0].shape[1])
cam.vfit_camera_visible_dynamic(ods, images=images, times_s=[t for t, _ in frames])

print(f"{len(frames)} frames inside the equilibrium's own time range")


The filter matters. The camera ran from 305 to 331 ms but the equilibrium only covers 316 to
327 ms, so a third of the frames have no equilibrium to overlay. Asking for one anyway would
silently snap to the nearest slice and draw a boundary from a different moment.

`theta_deg_range` is the other setting worth knowing: every overlay is swept toroidally, so
at the default sweep the flux surfaces project into a filled sheet. A narrow sweep is what
makes them read as nested contours.


In [ ]:
vaft.omas.plot_camera_visible_image_efit_overlay(
    ods, shot=39915, frame_index=10, theta_deg_range=(-8.0, 8.0)
)
plt.show()


## Interpretation Checkpoints

Work through these before moving on. Each has a definite answer in what you have already
plotted.

1. **The nine slices span 15 ms.** Look at the `q95` history. Does it rise or fall, and what
   does that tell you about the current profile as the discharge evolves?
2. **You derived beta and `li` rather than reading them.** What measurement would VEST need
   for the reconstruction to determine them directly, and which of the two shots has it?
3. **The `q` profile changes shape between `psi_norm` and `rho_tor_norm`.** Which regions
   stretch, and why would a transport study prefer the toroidal coordinate?
4. **The camera overlay shrinks between frames.** Cross-check it against the plasma-current
   history: are they telling the same story?
5. **Slice 8 has no boundary.** What would `plot_equilibrium_geometry_boundary(ods,
   time_slice=8)` do, and is refusing better than drawing something?


## Integrated Analysis

### Did the reconstruction actually solve the equation?

A converged reconstruction is not automatically a *consistent* one. EFIT reports a
convergence number — how little `psi` moved on its last iteration — but that says nothing
about whether the flux map it settled on satisfies the Grad-Shafranov equation with the
`p'` and `FF'` it also reports.

That is a separate question, and it can be measured directly: evaluate both sides and
compare.


In [ ]:
from vaft.data.eqdsk import read_geqdsk

efit = read_geqdsk(str(vaft.data.data_path("kineticEfit/g048224.00300"))).to_omas()
residual = vaft.omas.compute_grad_shafranov_residual(efit, time_slice=0)

print(f"points scored     : {int(residual.mask.sum())}")
print(f"median relative   : {float(np.nanmedian(residual.relative)):.4f}")


Note what that measurement had to be careful about. Outside the separatrix `psi` is not
monotonic — it turns around near the poloidal field coils and folds back into the same range
as the plasma. Scoring by flux value alone therefore re-admits vacuum grid points, where
`p'` and `FF'` are extrapolation and $\Delta^*\psi$ is reading the *coils*. On this shot that
would be a third of the points, and it inflates the answer by more than a decade.

VAFT masks on the boundary instead. The lesson generalises: **ask a question only where its
terms are defined.**

### What refinement changes

The same equilibrium, re-solved by CHEASE on a 513x513 fixed-boundary mesh, ships beside it.


In [ ]:
refined = read_geqdsk(str(vaft.data.data_path("kineticEfit/g048224.00300.chease"))).to_omas()
refined_residual = vaft.omas.compute_grad_shafranov_residual(refined, time_slice=0)

print(f"EFIT   median relative residual: {float(np.nanmedian(residual.relative)):.4f}")
print(f"CHEASE median relative residual: {float(np.nanmedian(refined_residual.relative)):.4f}")


In [ ]:
figure, axes = plt.subplots()
axes.semilogy(residual.psi_norm, residual.relative, label="EFIT reconstruction")
axes.semilogy(refined_residual.psi_norm, refined_residual.relative, label="CHEASE refined")
axes.set_xlabel(r"$\psi_N$")
axes.set_ylabel("relative Grad-Shafranov residual")
axes.legend()
axes.grid(alpha=0.3)
plt.show()


Both close their own equation to within a few percent, which is the honest headline: the
reconstruction was already consistent, and refinement did not rescue it from being wrong.

What refinement *did* buy is visible in the shape of the two curves. CHEASE's residual is
flat and featureless — the signature of a converged solve on a fine mesh. EFIT's wanders,
because its `psi` came from a fit to magnetic measurements rather than from solving the
equation, and the mismatch varies across the plasma.

So the question refinement answers is not "was the reconstruction wrong" but "what does this
equilibrium look like when the equation is solved to numerical precision, on a mesh fine
enough for a stability code to use".

### Same shape, same profiles

Refinement is only meaningful if it did not quietly change the equilibrium into a different
one. CHEASE reports that directly.


In [ ]:
efit_boundary = efit["equilibrium.time_slice.0.boundary.outline"]
refined_boundary = refined["equilibrium.time_slice.0.boundary.outline"]

figure, axes = plt.subplots()
axes.plot(efit_boundary["r"], efit_boundary["z"], label="EFIT boundary")
axes.plot(refined_boundary["r"], refined_boundary["z"], "--", label="CHEASE boundary")
axes.set_aspect("equal")
axes.set_xlabel("R [m]")
axes.set_ylabel("Z [m]")
axes.legend()
plt.show()


## Independent Exercise

### Vary the equilibrium and describe what changes

The cells below run CHEASE, so they need it installed and pointed at by the environment —
this is the session's **lab mode**. Uncomment and run them if you have it; read them if you
do not. Nothing above this point requires it.

`scan_chease` varies what CHEASE actually reads: the boundary, the pressure, and `FF'`. Each
variation is relative to the shot's own equilibrium, so `elongation_scale=1.10` means ten
percent more elongated than *this* discharge was.

One result is worth predicting before you run it. Scaling `FF'` up and down does **not** move
`li`, because CHEASE rescales the total current and a uniform factor is exactly the freedom
it normalises away. What moves `li` is the *shape* of `FF'` — hence `current_peaking`.


In [ ]:
# 1. Check whether CHEASE is available. Returns None if it is not configured.
# from vaft.code import find_chease_executable
# print(find_chease_executable())

# 2. Vary beta, li and the shape, and re-solve each case.
# from vaft.code import CHEASEConfig, EquilibriumVariation, scan_chease
#
# config = CHEASEConfig(nideal=6, nw=513, target_psin=0.993, relax=0.5)
# cases = scan_chease(
#     str(vaft.data.data_path("kineticEfit/g048224.00300")),
#     [
#         EquilibriumVariation("control"),
#         EquilibriumVariation("beta_up", pressure_scale=1.5),
#         EquilibriumVariation("li_up", current_peaking=1.0),
#         EquilibriumVariation("elong_up", elongation_scale=1.10),
#     ],
#     config=config,
#     workdir="outputs/03/scan",
# )

# 3. Derive each result and tabulate beta_N, li_3 and the elongation.
#    Which knob moved which quantity, and which left the others alone?
# for case in cases:
#     if not case.converged:
#         print(case.variation.label, "did not converge:", case.error)
#         continue
#     result = read_geqdsk(case.result.refined_geqdsk).to_omas()
#     vaft.omas.update_equilibrium_derived_profiles(result)
#     quantities = result["equilibrium.time_slice.0.global_quantities"]
#     print(case.variation.label,
#           float(quantities["beta_normal"]),
#           float(quantities["li_3"]))

# 4. Then ask the harder question: does each varied case still satisfy its own
#    Grad-Shafranov equation? Use compute_grad_shafranov_residual on each.


## Takeaways and Next Steps

- An equilibrium is a solution of the Grad-Shafranov equation, and whether a stored one
  *is* such a solution is a question you can ask rather than assume.
- A g-file carries six profiles. Volume, shape, beta and `li` are derived, and
  `update_equilibrium_derived_profiles` is what derives them — a refused plot is often an
  underived quantity.
- Reconstruction and forward modelling answer different questions. EFIT tells you what the
  discharge did; CHEASE tells you what a set of inputs implies, to numerical precision, on a
  mesh a stability code can use.
- A profile needs a coordinate, and the coordinate is part of the physics claim, not
  decoration.
- Ask a question only where its terms are defined. The same residual computed over vacuum
  grid points says something confident and wrong.

**Not covered here.** Straight-field-line coordinates — PEST, Boozer, Hamada — are the natural
next step for anyone heading toward stability analysis, and VAFT does not yet compute them:
the existing helper returns a geometric approximation rather than a flux-coordinate
transform. That gap is tracked as issue #472.

**Next**: Session 04 takes the equilibrium you now trust and looks at what fluctuates on top
of it.
